# 06 — Final evaluation and insurer / planner error costs

**Owner: Faris. Evaluation, confusion matrices, ranking confused classes, cost matrices and sensitivity analysis.**

**Functions:** `src/piece_of_crop/evaluate.py` (`load_predictions`, `make_cost_matrices`, `critical_matrix`, `cost_scenarios`, `paired_block_bootstrap`).

This notebook **does not train anything**. It reads held-out test predictions from `03` Random Forest, `04` CNN and `05` ResNet. Each CSV should contain `file, y_true, y_pred, confidence, p_0...p_9` and follow the same test split from `02`. Every dataset and prediction alignment is checked before comparison; malformed/misaligned results stop with an informative error.

**Outputs** under `results/`: `evaluation_metrics.csv`, `evaluation_per_class.csv`, `evaluation_confused_pairs.csv`, `evaluation_cost_sensitivity.csv`, `evaluation_paired_ci.csv`, `evaluation_per_class_gain.csv`, `evaluation_review.csv`, `evaluation_high_cost_errors.csv`, and confusion matrices and comparison figures in `results/figures/`.

**Cost matrix integrity:** The proposed insurer/planner numerical weights below are **illustrative severity points (not euros, premiums, or actuarially estimated costs)**. They must be reviewed by the team/domain expert. Because they are being drafted at evaluation time, do *not* claim they were pre-registered before training; label conclusions exploratory. None of these weights should be adjusted to make a preferred model win.

**Partial-run safe:** The notebook evaluates only *complete prediction CSVs that currently exist*. Set `STRICT_ALL_MODELS=True` **before final submission** to require all 18 expected files (3 models × RGB/13-band × seeds 0–2). Missing models are clearly displayed, never counted as zero. Bootstrap intervals are exploratory; holdout patches may be spatially correlated.

## 1. Load the canonical test split and discover model predictions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

from piece_of_crop import (CLASS_NAMES, MODEL_FILES, cost_scenarios, critical_matrix as make_critical_matrix,
                           find_project_root, load_predictions, make_cost_matrices, paired_block_bootstrap,
                           read_split)

SEEDS = (0, 1, 2)
STRICT_ALL_MODELS = False  # switch True for final deliverable after models 03,04,05 complete
BOOTSTRAPS = 500
RNG_SEED = 2026

project_root = find_project_root()
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"
predictions_dir = results_dir / "predictions"
models_dir = project_root / "models"
for folder in (results_dir, figures_dir, predictions_dir, models_dir):
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)

split = read_split(results_dir / "split.csv")
reference = split.loc[split["split"].eq("test")].copy().reset_index(drop=True)
reference["label"] = reference["label"].astype(int)
label_ids = np.arange(len(CLASS_NAMES), dtype=int)
if not np.array_equal(np.sort(reference["label"].unique()), label_ids):
    raise ValueError("Test data must contain all ten classes.")
if reference["file"].duplicated().any():
    raise ValueError("Test filenames are not unique.")
print("Test set:", len(reference), "patches; classes:", list(CLASS_NAMES))

# Reads every prediction CSV and checks it has the same test files and labels as split.csv.
predictions, missing = load_predictions(predictions_dir, reference, SEEDS)
print("Prediction sets found:", len(predictions), "/ 18")
print("Missing:", missing if missing else "none")
if STRICT_ALL_MODELS and missing:
    raise FileNotFoundError("Final evaluation requires all 18 prediction CSVs. Missing: " + ", ".join(missing))
if not predictions:
    raise FileNotFoundError("No prediction CSVs yet. Run notebook 03, then 04 and 05.")

## 2. Define and save cost matrices *before inspecting metrics*

These cost **severity points are scenario assumptions** (0 for correct labels, roughly 1–10 for incorrect labels). They represent decision-support risk rather than money. **Insurer**: crop/pasture/herbaceous confusions are more important; crop ↔ built-up confusion also matters. **Planner**: natural/green versus built-up land gets more emphasis. The rules are written as code in `make_cost_matrices` (`src/piece_of_crop/evaluate.py`), so they can be read and tested. The matrices are asymmetric where appropriate and saved to `results/insurer_cost_matrix.csv` and `results/planner_cost_matrix.csv`.

Insurer-critical directed errors are separately defined from the cost weights: AnnualCrop↔PermanentCrop, agricultural/crop/pasture↔HerbaceousVegetation, crop/pasture↔built-up or Forest. They will be counted per test sample.

In [ ]:
cost_matrices = make_cost_matrices()
for name, matrix in cost_matrices.items():
    frame = pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES)
    frame.to_csv(results_dir / f"{name}_cost_matrix.csv", index_label="actual_class")
    print(f"\n{name.upper()} cost matrix (severity points; rows=actual, cols=predicted):")
    display(frame)

critical_matrix = make_critical_matrix()
print("Critical directed confusion pairs:", int(critical_matrix.sum()))

## 3. Test accuracy, macro-F1, per-class metrics and cost-weighted error

Cost-weighted error is **mean severity points per test patch**: `mean(C[y_true, y_pred])`. Critical-error rate is the fraction of *all* test patches with a predefined critical confusion, not the fraction of incorrect predictions. Both can reveal different model rankings from accuracy. Results are stored per model/input/seed; averages are reported only across available seeds.

In [ ]:
metric_rows=[]
class_rows=[]
confusions={}
for (model,input_type,seed), df in predictions.items():
    truth = df['y_true'].to_numpy(dtype=int)
    predicted = df['y_pred'].to_numpy(dtype=int)
    cm = confusion_matrix(truth,predicted,labels=label_ids)
    confusions[(model,input_type,seed)] = cm
    prec,rec,f1,support = precision_recall_fscore_support(
        truth,predicted,labels=label_ids,zero_division=0)
    errors = truth!=predicted
    critical = critical_matrix[truth,predicted]
    row=dict(model=model,input=input_type,seed=seed,n_test=len(df),
             accuracy=accuracy_score(truth,predicted),
             macro_f1=f1_score(truth,predicted,labels=label_ids,average='macro',zero_division=0),
             error_rate=errors.mean(), critical_error_rate=critical.mean(),
             critical_share_of_errors=float(critical.sum()/max(1,errors.sum())))
    for name,mat in cost_matrices.items():
        row[f'{name}_cost'] = mat[truth,predicted].mean()
    metric_rows.append(row)
    for cls in label_ids:
        class_rows.append({'model':model,'input':input_type,'seed':seed,
                           'class_name':CLASS_NAMES[cls], 'class_id':int(cls),
                           'support':int(support[cls]), 'precision':prec[cls],
                           'recall':rec[cls], 'f1':f1[cls]})
metrics = pd.DataFrame(metric_rows).sort_values(['model','input','seed'])
per_class = pd.DataFrame(class_rows).sort_values(['model','input','seed','class_id'])
metrics.to_csv(results_dir/'evaluation_metrics.csv',index=False)
per_class.to_csv(results_dir/'evaluation_per_class.csv',index=False)
print('Test metrics per run:')
display(metrics.round(4))
print('Mean and standard deviation across available seeds:')
display(metrics.groupby(['model','input'])[
    ['accuracy','macro_f1','critical_error_rate','insurer_cost','planner_cost']]
    .agg(['mean','std']).round(4))

## 4. Confusion matrices and ranked confusion pairs

For **every available seed** save a 10×10 confusion matrix (rows actual, columns predicted). Also aggregate directed errors `(true -> predicted)` and symmetric unordered pairs within each model/input for a useful ranking. The ranking table is derived from model predictions, not manually supplied values.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
pair_rows=[]
for (model,input_type,seed), cm in confusions.items():
    fig,ax=plt.subplots(figsize=(10,9))
    ConfusionMatrixDisplay(cm,display_labels=CLASS_NAMES).plot(
        ax=ax,xticks_rotation=65,values_format='d',colorbar=False)
    ax.set_title(f'{model} / {input_type} / seed {seed} (test)')
    fig.tight_layout()
    fig.savefig(figures_dir/f'evaluation_confusion_{model}_{input_type}_seed{seed}.png',
                dpi=150,bbox_inches='tight')
    plt.close(fig)
    for i in label_ids:
        for j in label_ids:
            if i!=j and cm[i,j]>0:
                pair_rows.append({'model':model,'input':input_type,'seed':seed,
                                  'actual':CLASS_NAMES[i], 'predicted':CLASS_NAMES[j],
                                  'n_errors':int(cm[i,j]), 'critical':bool(critical_matrix[i,j])})
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(results_dir/'evaluation_confused_pairs.csv',index=False)
print('Most frequent directed errors (counts summed across seeds, not distinct patches):')
if not pairs.empty:
    ranking = (pairs.groupby(['model','input','actual','predicted','critical'],as_index=False)
               ['n_errors'].sum().sort_values(['model','input','n_errors'],ascending=[True,True,False]))
    display(ranking.groupby(['model','input'],group_keys=False).head(10))
    pairs2 = pairs.copy()
    pairs2['unordered_pair'] = [ ' ↔ '.join(sorted([a,b])) for a,b in zip(pairs2['actual'],pairs2['predicted']) ]
    symmetric = pairs2.groupby(['model','input','unordered_pair'],as_index=False)['n_errors'].sum()
    print('Most confused unordered class pairs:')
    display(symmetric.sort_values('n_errors',ascending=False)
            .groupby(['model','input'],group_keys=False).head(10))
else:
    print('No misclassifications in available predictions.')
print('Saved confusion matrices:',len(confusions))

## 5. Compare RGB vs multispectral per class and across seeds

The `13band - rgb` difference is computed **only for paired seeds from the same model**. A positive per-class recall difference means multispectral classification recovered more true examples of that class on the shared test set. Three seeds do not by themselves give a robust uncertainty estimate.

In [ ]:
per_class_gain_rows=[]
for model in MODEL_FILES:
    for seed in SEEDS:
        a = per_class.query('model == @model and input == "rgb" and seed == @seed')
        b = per_class.query('model == @model and input == "13band" and seed == @seed')
        if len(a)!=10 or len(b)!=10:
            continue
        pair = a.merge(b,on=['model','seed','class_id','class_name','support'],suffixes=('_rgb','_13band'))
        for _,r in pair.iterrows():
            per_class_gain_rows.append({'model':model,'seed':seed,'class_name':r['class_name'],
                                        'precision_gain':r['precision_13band']-r['precision_rgb'],
                                        'recall_gain':r['recall_13band']-r['recall_rgb'],
                                        'f1_gain':r['f1_13band']-r['f1_rgb']})
per_class_gain = pd.DataFrame(per_class_gain_rows,columns=['model','seed','class_name','precision_gain','recall_gain','f1_gain'])
per_class_gain.to_csv(results_dir/'evaluation_per_class_gain.csv',index=False)
if not per_class_gain.empty:
    display(per_class_gain.groupby(['model','class_name'])[['precision_gain','recall_gain','f1_gain']]
            .mean().round(4))
else:
    print('No paired RGB/13-band seed results yet.')

## 6. Sensitivity analysis: check conclusions under alternative cost assumptions

The main claim is tested with: **all errors equally weighted**, **baseline insurer**, **insurer-critical mistakes half as severe**, **insurer-critical mistakes twice as severe**, and **planner**. These are *stress tests of ordinal severity assumptions*, not estimated financial losses. It is legitimate for a model's ranking to change; do not suppress those results.

In [ ]:
scenarios = cost_scenarios(cost_matrices, critical_matrix)
sensitivity_rows=[]
for (model,input_type,seed),df in predictions.items():
    t = df['y_true'].to_numpy(dtype=int)
    p = df['y_pred'].to_numpy(dtype=int)
    for scenario,mat in scenarios.items():
        sensitivity_rows.append({'model':model,'input':input_type,'seed':seed,
                                 'scenario':scenario,'cost_weighted_error':float(mat[t,p].mean())})
sensitivity=pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(results_dir/'evaluation_cost_sensitivity.csv',index=False)
print('Mean cost per test patch across available seeds:')
display(sensitivity.groupby(['scenario','model','input'])['cost_weighted_error']
        .agg(['mean','std']).round(4))

summary = metrics.groupby(['model','input'])[['accuracy','macro_f1','insurer_cost','planner_cost']].agg(['mean','std'])
summary.to_csv(results_dir/'evaluation_comparison_summary.csv')
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,metric in zip(axes,('macro_f1','insurer_cost')):
    panel=metrics.pivot_table(index='model',columns='input',values=metric,aggfunc='mean')
    panel.reindex(columns=['rgb','13band']).plot(kind='bar',ax=ax,rot=0)
    ax.set_title(metric.replace('_',' ').title())
    ax.set_xlabel('Model')
    ax.grid(axis='y',alpha=.25)
fig.tight_layout()
fig.savefig(figures_dir/'evaluation_model_comparison.png',dpi=150,bbox_inches='tight')
plt.show()

## 7. Paired uncertainty estimates for RGB vs 13-band

For each model and matching seed we compute per-test-patch cost differences: `(13band cost - RGB cost)`. Negative = multispectral is better. A **paired block bootstrap** samples geographic blocks from the held-out test set (if the `block` column exists) and calculates a 95% interval for the *mean difference across common seeds*. This accounts for within-block correlation more conservatively than independent patch resampling, but does **not** fully capture scene-level or future-deployment uncertainty. With only 3 seeds, the seed standard deviation is reported separately. If block labels are absent, it falls back to patch resampling and clearly labels the method.

In [ ]:
bootstrap_rows=[]
bootstrap_rng=np.random.default_rng(RNG_SEED)
bootstrap_groups=reference['block'].astype(str).to_numpy() if 'block' in reference else np.arange(len(reference)).astype(str)
print('CI resampling unit:', 'spatial block' if 'block' in reference else 'individual patch',
      '| groups:',len(np.unique(bootstrap_groups)))

for model in MODEL_FILES:
    common_seeds=[s for s in SEEDS if (model,'rgb',s) in predictions and (model,'13band',s) in predictions]
    if not common_seeds:
        continue
    for cost_name,cost_matrix in cost_matrices.items():
        diffs=[]
        for seed in common_seeds:
            rgb=predictions[(model,'rgb',seed)]
            ms=predictions[(model,'13band',seed)]
            if not np.array_equal(rgb['file'].to_numpy(),ms['file'].to_numpy()):
                raise ValueError('Paired input files are not aligned')
            t=rgb['y_true'].to_numpy(dtype=int)
            diff=(cost_matrix[t,ms['y_pred'].to_numpy(dtype=int)] -
                  cost_matrix[t,rgb['y_pred'].to_numpy(dtype=int)])
            diffs.append(diff)
        diffs=np.vstack(diffs)                 # (seed, image)
        mean_by_patch=diffs.mean(axis=0)       # pair same image, average common seeds
        ci_low, ci_high, n_units = paired_block_bootstrap(mean_by_patch, bootstrap_groups, BOOTSTRAPS, bootstrap_rng)
        seed_means=diffs.mean(axis=1)
        bootstrap_rows.append({'model':model,'cost_matrix':cost_name,
                               'n_seeds':len(common_seeds),'mean_13band_minus_rgb':seed_means.mean(),
                               'seed_std':float(seed_means.std(ddof=1)) if len(seed_means)>1 else np.nan,
                               'ci_low_95':ci_low,
                               'ci_high_95':ci_high,
                               'resampling_unit':'block' if 'block' in reference else 'patch',
                               'n_units':n_units})
paired_ci = pd.DataFrame(bootstrap_rows)
paired_ci.to_csv(results_dir/'evaluation_paired_ci.csv',index=False)
if not paired_ci.empty:
    display(paired_ci.round(4))
else:
    print('No fully paired RGB/13-band results to bootstrap.')

## 8. Misclassified cases and low-confidence human review

Export the 20 highest-insurer-severity misclassifications from seed 0 for manual inspection. If probabilities/confidence are available, also report a **fixed 0.60 confidence threshold** for *review coverage* (how many predictions are sent to a human), and recall of critical errors by review. This is an illustrative rule, **not a tuned or calibrated production policy**; compare it without claiming clinical/actuarial validation.

In [ ]:
errors_rows=[]
review_rows=[]
REVIEW_CONFIDENCE_THRESHOLD = 0.60
for (model,input_type,seed),df in predictions.items():
    t=df['y_true'].to_numpy(dtype=int)
    p=df['y_pred'].to_numpy(dtype=int)
    tmp=df[['file','y_true','y_pred']].copy()
    tmp['model'],tmp['input'],tmp['seed'] = model,input_type,seed
    tmp['actual_name']=[CLASS_NAMES[k] for k in t]
    tmp['predicted_name']=[CLASS_NAMES[k] for k in p]
    tmp['insurer_cost'] = cost_matrices['insurer'][t,p]
    tmp['planner_cost'] = cost_matrices['planner'][t,p]
    tmp['critical'] = critical_matrix[t,p]
    tmp=tmp.loc[t!=p]
    errors_rows.append(tmp)
    if 'confidence' in df:
        confidence=df['confidence'].to_numpy(dtype=float)
        if not np.isfinite(confidence).all() or np.any((confidence<0)|(confidence>1)):
            raise ValueError(f'{model}/{input_type}/{seed}: invalid confidence values')
        flag = confidence<REVIEW_CONFIDENCE_THRESHOLD
        critical=critical_matrix[t,p]
        review_rows.append({'model':model,'input':input_type,'seed':seed,
                            'threshold':REVIEW_CONFIDENCE_THRESHOLD,'review_fraction':flag.mean(),
                            'critical_errors_found_fraction':float((flag&critical).sum()/max(1,critical.sum())),
                            'accuracy_not_flagged':float((p[~flag]==t[~flag]).mean()) if (~flag).any() else np.nan})

high_cost_errors=pd.concat(errors_rows,ignore_index=True) if errors_rows else pd.DataFrame()
high_cost_errors=high_cost_errors.sort_values(['insurer_cost','model','seed'],ascending=[False,True,True])
high_cost_errors.to_csv(results_dir/'evaluation_high_cost_errors.csv',index=False)
review=pd.DataFrame(review_rows)
review.to_csv(results_dir/'evaluation_review.csv',index=False)
print('Top 20 misclassifications for manual review (counts across runs may repeat images):')
display(high_cost_errors.head(20))
if not review.empty:
    print('Human-review threshold analysis:')
    display(review.round(4))
print('Evaluation complete for',len(predictions),'available prediction CSVs.')
print('Missing expected files:', missing if missing else 'none')

## 9. Reporting guidance (do not overclaim)

1. Primary finding should compare *paired* RGB and 13-band test metrics and **severity-weighted errors**, not accuracy alone.
2. Report the complete class-pair ranking and at least a few real misclassified patches. The `evaluation_high_cost_errors.csv` provides filenames; inspect the actual GeoTIFFs with labels before writing an error story.
3. Report per-seed variation and paired confidence intervals with caveats: spatial leakage remains possible, block sizes are approximate, class labels may be mixed, and 3 random seeds are few.
4. **Do not call the illustrative severity weights real insurance costs**. Obtain expert input before recommending commercial deployment. Decisions must include human review.
5. Run with `STRICT_ALL_MODELS=True` for the final deliverable; partial runs are for development only. Do not merge incomplete findings into the final report.